# OSR-303 Clock Fault Injection Example

## Connection Diagram

Use [powershorter](https://gitee.com/osr-tech/powershorter) to drive the CycleWarper for clock fault injection on the OSR-303 board, and use a PICO3206D oscilloscope to observe the glitch.

<img src="images/clk-303.jpg"  width="800">

<img src="images/cyclewarper.png"  width="800">

**Clock fault disturbance can prevent the 303 board from operating normally. We use the PowerShorter's GPIO to reset the board; the IO outputs a low level to reset the board.**

## Download the Project

As shown in `https://gitee.com/osr-tech/osr-303/blob/master/project/STM32CubeIDE-303Guide.md`, we use STM32CubeIDE to download the `FORLOOP` project (in the project folder).

## Communication Test

First set the clock switch to internal, then press the reset button once to perform a communication test.

In [1]:
import serial

In [2]:
loopv = 201

In [3]:
toe = serial.Serial('/dev/cu.usbmodem21302', 115200, timeout=1)

In [4]:
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

201


## Controlling the 303 Reset

To control the device reset, use the `GPIO` of `powershorter` to control the chip's RESET.

In [5]:
import power_shorter as ps
import time

In [6]:
clk_dev = ps.CycleWarper('/dev/cu.usbserial-2140') # select the serial port

In [7]:
def reset_toe(): 
    clk_dev.relay(ps.GPIO.GPIO1, 0)
    time.sleep(0.3)
    clk_dev.relay(ps.GPIO.GPIO1, 1)
    time.sleep(1)
    toe.reset_input_buffer()

In [9]:
# Test the reset functionality
reset_toe()  
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

201


In [22]:
clk_dev.engine_cfg?

Signature:
clk_dev.engine_cfg(
    engine: power_shorter.ctrl.Engine,
    delay: int,
    pulse: int,
    trigger_mode: power_shorter.ctrl.TRIGGER_MODE = <TRIGGER_MODE.RISE: 0>,
    trigger_edges: int = 1,
)
Docstring:
set trigger and delay for power shorter engine, to control CycleWarper

Args:
    engine (Engine): which engine to be set.
    delay (int): delay time from trigger in unit of 10ns.
    pulse (int): pulse width in unit of 10ns.
    trigger_mode: (TRIGGER_MODE): trigger mode. Defaults to TRIGGER_MODE.RISE.
    trigger_edges (int, optional): how many edges as trigger event. Defaults to 1.
File:      ~/Gitlab/atlasv2/py313/lib/python3.13/site-packages/power_shorter/ctrl.py
Type:      method

## Clock Fault Injection

In [10]:
def glitch(delay, pulse):
    clk_dev.engine_cfg(ps.Engine.E1, delay, pulse, trigger_mode=ps.TRIGGER_MODE.RISE, trigger_edges=1) # on receiving the trigger signal, wait delay*10 ns, then generate a faulty clock for pulse*10 ns
    clk_dev.arm(ps.Engine.E1)
    toe.write(loopv.to_bytes(1, 'little'))
    ret = toe.read(1)
    retv = int.from_bytes(ret, 'little')
    state = None
    if ret == b'':
        state  = 'dead'
        reset_toe()
    elif retv == loopv:
        state = 'normal'
    else:
        state = 'glitch success'
        #print(state, retv, delay, pulse)
    return state, retv, delay, pulse

In [13]:
glitch(1, 500)

('normal', 201, 1, 500)

The pico oscilloscope observes the trigger signal output by the board. You can observe the normal run time and the fault-injection run time; because a higher-frequency faulty clock is introduced, the run time becomes shorter.
<figure style="text-align:center">
  <img src="./images/normal_clk_pico.png" width="400">
  <figcaption><strong>Normal clock</strong></figcaption>
</figure>

<figure style="text-align:center">
  <img src="./images/glitch_clk_pico.png" width="400">
  <figcaption><strong>Faulty clock</strong></figcaption>
</figure>

## Visualizing Fault Parameters and Results

You can conveniently observe fault parameters and results using [FaultViz](https://gitee.com/osr-tech/faultviz).

In [14]:
import faultviz

In [15]:
faultviz.start_view_service()

# Starting io.deephaven.python.server.EmbeddedServer
deephaven.cacheDir=/Users/ping/Library/Caches/io.Deephaven-Data-Labs.deephaven
deephaven.configDir=/Users/ping/Library/Application Support/io.Deephaven-Data-Labs.deephaven
deephaven.dataDir=/Users/ping/Library/Application Support/io.Deephaven-Data-Labs.deephaven
# io.deephaven.internal.log.LoggerFactoryServiceLoaderImpl: searching for 'io.deephaven.internal.log.LoggerFactory'...
# io.deephaven.internal.log.LoggerFactoryServiceLoaderImpl: found 'io.deephaven.internal.log.LoggerFactorySlf4j'
Server started on port 12345


In [16]:
vt = faultviz.ViewWidget()

In [17]:
state, retv, delay, pulse = glitch(3200, 20)
vt.update(state=state, val=retv, delay=delay, pulse=pulse)

In [18]:
vt.show()

DeephavenWidget(height=150, iframe_url='http://localhost:12345/iframe/table/?name=_31fbd860_8fd6_4338_b383_563…

DeephavenWidget(height=600, iframe_url='http://localhost:12345/iframe/table/?name=_45b24429_d088_472a_b3ee_19c…

## Random-Parameter Injection

In [19]:
from tqdm.notebook import tnrange
import random

>During clock fault injection, rotate the CycleWarper's frequency adjustment knob to set the normal operating frequency (Normal) and the faulty operating frequency (Glitch). Press the knob to switch between the two frequencies.

<img src="./images/clk_set.png" width="400">

In [21]:
for i in tnrange(1000):
    delay=random.randint(2000, 4000)
    pulse=random.randint(3000, 5000)
    state, retv, delay, pulse = glitch(delay, pulse)
    vt.update(state=state, val=retv, delay=delay, pulse=pulse)

  0%|          | 0/1000 [00:00<?, ?it/s]

## Results

![image](images/clk-faultviz.png)